# Aprendizagem de Máquina para Desenvolvedores Web

## Web Academy



## Parte 1: O Ciclo de Vida do ML em Produção (Fundamento Conceitual)

### 1.1 Da Exploração à Produção: Por que o Jupyter Notebook não é o ambiente final


O desenvolvimento de um sistema de Machine Learning é dividido em duas fases distintas e com propósitos fundamentalmente diferentes: a **Fase Experimental** e a **Fase de Produção**. Compreender essa divisão é o primeiro passo para construir sistemas de ML robustos.

**Os Dois Mundos do ML**

A **Fase Experimental** é onde a ciência de dados acontece. É um processo iterativo de exploração, visualização, pré-processamento de dados e teste de diferentes algoritmos para encontrar um modelo promissor.
 - A ferramenta predominante nesta fase é o Jupyter Notebook, e por boas razões. Sua natureza interativa, que permite a execução de código em células isoladas e a visualização imediata de resultados, é inigualável para a prototipagem rápida e a análise exploratória de dados.

A **Fase de Produção**, por outro lado, foca na engenharia. O objetivo aqui é pegar o modelo e os passos de pré-processamento validados na fase anterior e transformá-los em um serviço confiável, automatizado, escalável e monitorável.
 - É nesta transição que as mesmas características que tornam os notebooks excelentes para a experimentação se tornam passivos críticos.

**A Armadilha do Jupyter em Produção**

A tentativa de usar notebooks diretamente em ambientes de produção leva a uma série de problemas sistêmicos que comprometem a confiabilidade e a manutenibilidade do software:

* **Reprodutibilidade e Estado Oculto:** A maior força do notebook — a execução de células fora de ordem — é sua maior fraqueza em produção. É possível criar variáveis, treinar modelos e alterar o estado do kernel de forma não linear. Isso resulta em um "estado oculto", tornando quase impossível garantir que o notebook executará da mesma forma duas vezes. Um sistema de produção, por definição, exige execuções consistentes e reprodutíveis.
* **Controle de Versão Inviável:** Notebooks são armazenados como arquivos JSON complexos que misturam código, saídas de texto, imagens e metadados. Ao visualizar um `diff` em um sistema de controle de versão como o Git, as alterações são praticamente ilegíveis. Isso torna a colaboração em equipe, as revisões de código (code reviews) e a fusão de branches (merging) tarefas extremamente difíceis e propensas a erros.
* **Ausência de Boas Práticas de Engenharia:** O ambiente do notebook não se presta naturalmente a práticas de engenharia de software essenciais. A implementação de testes unitários, integração contínua (CI/CD), gerenciamento explícito de dependências e um tratamento de erros robusto é, na melhor das hipóteses, complicada. Notebooks incentivam a escrita de código no estilo de script, em vez de software modular e testável.

**A Mudança de Mentalidade para Produção**

O conflito fundamental não é técnico, mas filosófico. Um notebook é um ambiente de desenvolvimento interativo (IDE) projetado para a exploração. Sua filosofia prioriza o feedback imediato e a experimentação não linear. Um sistema de produção, no entanto, requer a definição de um processo declarativo e automatizável. Ele precisa de um script que possa ser executado de cima a baixo, de forma repetível, por um sistema automatizado. A transição para a produção exige a adoção de uma mentalidade de engenharia, movendo-se de um *script interativo e imperativo* (o notebook) para um *pipeline reproduzível e automatizado* (o código de produção).

### 1.2 Serialização de Modelos: Salvando o estado de um modelo treinado com `joblib`

Após o treinamento, um modelo de machine learning existe como um objeto na memória do Python, com todos os seus parâmetros e estados aprendidos. Para usar este modelo no futuro sem a necessidade de retreiná-lo, é preciso persistir esse estado em um arquivo. Esse processo é chamado de **serialização**.

**O que é Serialização?**

Serialização é o processo de converter um objeto Python em memória em um fluxo de bytes que pode ser facilmente armazenado em um arquivo ou transmitido por uma rede. O processo inverso, de carregar o fluxo de bytes de volta para um objeto Python, é chamado de desserialização.

**`pickle` vs. `joblib`**

A biblioteca padrão do Python para essa tarefa é o `pickle`. No entanto, para objetos do ecossistema científico do Python, especialmente aqueles que contêm grandes arrays NumPy (como os modelos do Scikit-learn), a biblioteca `joblib` é a escolha recomendada. `Joblib` oferece uma serialização mais eficiente para esses tipos de objetos, resultando em arquivos menores e tempos de carregamento mais rápidos.

**A Lição Crítica: Serializar o Pipeline *Inteiro***

Este é talvez o conceito mais crucial para a operacionalização de modelos de NLP. Um erro comum e catastrófico de iniciantes é treinar um vetorizador de texto (como o `TfidfVectorizer`) e um modelo classificador (como o `LogisticRegression`) separadamente, e salvar apenas o objeto do classificador treinado.

Isso falhará em produção. O `TfidfVectorizer` aprende um vocabulário específico e calcula os pesos IDF (Inverse Document Frequency) com base *exclusivamente* nos dados de treinamento. O modelo `LogisticRegression` é então treinado em uma matriz numérica gerada por *esse vetorizador específico e seu estado aprendido*. Se em produção apenas o modelo for carregado e um novo vetorizador for instanciado, este novo vetorizador não terá o mesmo vocabulário ou os mesmos pesos IDF. O espaço de características será completamente diferente, e as previsões do modelo serão, na melhor das hipóteses, aleatórias.

A solução elegante e robusta para este problema é a classe `Pipeline` do Scikit-learn. Um `Pipeline` encadeia múltiplos passos de transformação e um estimador final (o modelo) em um único objeto composto.

A regra de ouro é: **não se salva o modelo, salva-se o pipeline treinado inteiro**. Este único artefato serializado contém todo o estado necessário — o vocabulário, os pesos IDF do vetorizador e os coeficientes do modelo — para ir do texto bruto à previsão de forma consistente e reprodutível.

### 1.3 Expondo o Modelo via API REST: O padrão de mercado

Uma vez que o pipeline treinado está salvo, ele precisa ser exposto de uma forma que outras aplicações possam consumi-lo. O padrão da indústria para isso é encapsular o modelo em um microsserviço e expô-lo através de uma API REST.

**Por que uma API? Desacoplamento e Microsserviços**

Tratar o modelo de ML como um microsserviço oferece um desacoplamento crucial. A equipe de frontend que desenvolve uma aplicação web não precisa saber Python ou se preocupar com as dependências do modelo. Eles simplesmente precisam fazer uma requisição HTTP POST para um endpoint como `/predict` com os dados necessários. Isso permite que o ciclo de vida de desenvolvimento e implantação do serviço de ML seja independente das aplicações clientes, promovendo agilidade e especialização das equipes.

**Frameworks de API em Python: Foco em FastAPI**

Existem vários frameworks web em Python, mas para a tarefa de servir modelos de ML, o FastAPI se destaca como a escolha moderna e de alto desempenho.

* **Performance Excepcional:** Construído sobre o ASGI (Asynchronous Server Gateway Interface), em vez do tradicional WSGI, o FastAPI é um dos frameworks Python mais rápidos disponíveis, com desempenho comparável ao de NodeJS e Go.
* **Validação de Dados com Pydantic:** Ao definir a estrutura dos dados de entrada esperados usando uma simples classe Pydantic, o FastAPI valida automaticamente todas as requisições recebidas. Se uma requisição não estiver no formato correto, o FastAPI a rejeita com uma resposta de erro `422 Unprocessable Entity` clara e detalhada.
* **Documentação Interativa Automática (Swagger UI):** Apenas escrevendo o código com anotações de tipo padrão do Python, o FastAPI gera automaticamente uma documentação de API interativa e completa, acessível em `/docs`. Esta interface, conhecida como Swagger UI, permite que os desenvolvedores explorem e testem os endpoints diretamente do navegador.

| Característica | Flask | Django | FastAPI |
| :--- | :--- | :--- | :--- |
| **Tipo** | Micro-framework | Framework Full-stack | Micro-framework |
| **Performance** | Boa (WSGI Síncrono) | Moderada (WSGI Síncrono) | Excelente (ASGI Assíncrono) |
| **Suporte Assíncrono** | Limitado (via extensões) | Parcial (adicionado recentemente) | Nativo e Central |
| **Validação de Dados** | Manual (requer bibliotecas) | Integrada (Forms/Serializers) | Automática e Nativa (Pydantic) |
| **Auto-Documentação** | Manual (requer extensões) | Manual (requer extensões) | Automática e Nativa (OpenAPI) |
| **Caso de Uso Ideal para ML** | Prototipagem rápida, APIs simples | Aplicações web complexas com ML integrado | Microsserviços de ML de alta performance |

---

## Parte 2: Passo a Passo da Prática: Finalizando e Salvando o Pipeline

### 2.1 Revisitar o código da análise de sentimento

Este é um código típico da fase experimental. Ele funciona, mas é frágil, pois depende da gestão manual de dois objetos separados (`vectorizer` e `model`).

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Dados de exemplo
data = {
    'review': [
        'This movie was fantastic, I loved it!',
        'A complete waste of time, the plot was terrible.',
        'An amazing film with brilliant acting.',
        'I would not recommend this to anyone.',
        'The best movie of the year, hands down.',
        'It was so boring I fell asleep halfway through.',
        'A masterpiece of cinema.',
        'The script was predictable and unoriginal.',
        'I was on the edge of my seat the entire time!',
        'Horrible from start to finish.'
    ],
    'sentiment': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] # 1 para Positivo, 0 para Negativo
}
df = pd.DataFrame(data)

# Dividir os dados para demonstração
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

# Passo 1: Vetorização (separada)
print("Vetorizando o texto...")
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Passo 2: Treinamento do modelo (separado)
print("Treinando o modelo...")
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# Avaliação
print("Avaliando o modelo...")
y_pred = model.predict(X_test_vec)
print(f"Acurácia no conjunto de teste: {accuracy_score(y_test, y_pred):.2f}")

Vetorizando o texto...
Treinando o modelo...
Avaliando o modelo...
Acurácia no conjunto de teste: 0.50


### 2.2 Usar a classe `Pipeline` do Scikit-learn

Agora, vamos refatorar o código para usar a classe `Pipeline`, unindo o vetorizador e o classificador em um único objeto. Esta é a abordagem correta para a produção.

In [17]:
from sklearn.pipeline import Pipeline

# Definir os passos do pipeline
# Cada passo é uma tupla com um nome (string) e o objeto estimador
steps = [
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression())
]

# Criar o pipeline
sentiment_pipeline = Pipeline(steps)

print("Pipeline criado:")
print(sentiment_pipeline)

Pipeline criado:
Pipeline(steps=[('vectorizer', TfidfVectorizer()),
                ('classifier', LogisticRegression())])


### 2.3 Treinar este pipeline com todos os dados

Para o artefato final de produção, é uma prática comum retreinar o modelo escolhido em todo o conjunto de dados disponível. Isso permite que o modelo aprenda com o máximo de informações possível antes de ser implantado.

In [18]:
# Usar todos os dados para o treinamento final
all_reviews = df['review']
all_labels = df['sentiment']

print("\nTreinando o pipeline final com todos os dados...")
sentiment_pipeline.fit(all_reviews, all_labels)
print("Pipeline treinado com sucesso!")

# Testando o pipeline treinado
test_review = "This was a great movie, truly excellent."
prediction = sentiment_pipeline.predict([test_review])
print(f"\nTeste com uma nova review: '{test_review}'")
print(f"Predição: {'Positivo' if prediction[0] == 1 else 'Negativo'}")


Treinando o pipeline final com todos os dados...
Pipeline treinado com sucesso!

Teste com uma nova review: 'This was a great movie, truly excellent.'
Predição: Positivo


### 2.4 Salvar o pipeline treinado com `joblib.dump()`

Agora que temos nosso pipeline treinado, o passo final é serializá-lo para um arquivo. O arquivo `sentiment_pipeline.joblib` conterá tudo o que é necessário para fazer previsões: a lógica de pré-processamento, o vocabulário aprendido e os pesos do modelo.

In [19]:
import joblib
import os

# Criar o diretório se não existir
os.makedirs('models', exist_ok=True)

# Salvar o pipeline
file_path = 'models/sentiment_pipeline.joblib'
joblib.dump(sentiment_pipeline, file_path)

print(f"\nPipeline salvo com sucesso em: {file_path}")

# Verificação (opcional): Carregar e testar novamente
loaded_pipeline = joblib.load(file_path)
test_review_2 = "A terribly boring film."
prediction_loaded = loaded_pipeline.predict([test_review_2])
print(f"\nTeste com o pipeline carregado: '{test_review_2}'")
print(f"Predição: {'Positivo' if prediction_loaded[0] == 1 else 'Negativo'}")


Pipeline salvo com sucesso em: models/sentiment_pipeline.joblib

Teste com o pipeline carregado: 'A terribly boring film.'
Predição: Negativo


### 2.5 Exercício 1: Construa e Salve um Pipeline (15 min)

**Tarefa:** O `TfidfVectorizer` considera a frequência dos termos. Uma alternativa mais simples é o `CountVectorizer`, que apenas conta as ocorrências de cada palavra (um modelo "Bag-of-Words").

1.  Importe o `CountVectorizer` de `sklearn.feature_extraction.text`.
2.  Crie um novo pipeline que use o `CountVectorizer` em vez do `TfidfVectorizer`. Mantenha o `LogisticRegression` como classificador.
3.  Treine este novo pipeline com todos os dados (`all_reviews`, `all_labels`).
4.  Salve o pipeline treinado em um arquivo chamado `models/bow_pipeline.joblib`.

In [20]:
# Solução do Exercício 1
from sklearn.feature_extraction.text import CountVectorizer

# 1. e 2. Crie o novo pipeline
bow_pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', LogisticRegression())
])

# 3. Treine o pipeline
bow_pipeline.fit(all_reviews, all_labels)

# 4. Salve o pipeline
bow_file_path = 'models/bow_pipeline.joblib'
joblib.dump(bow_pipeline, bow_file_path)

print(f"Pipeline Bag-of-Words salvo com sucesso em: {bow_file_path}")

Pipeline Bag-of-Words salvo com sucesso em: models/bow_pipeline.joblib


---

## Parte 3: Passo a Passo da Prática: Construindo a API com FastAPI

### 3.1 Estruturar um projeto simples

Vamos criar os arquivos e diretórios necessários para nossa API diretamente do notebook usando comandos mágicos (`%%writefile`).

A estrutura será:
```
sentiment_api/
├── app/
│   ├── __init__.py
│   └── main.py
├── models/
│   └── sentiment_pipeline.joblib
└── requirements.txt
```

In [21]:
# Criando os diretórios
os.makedirs('app', exist_ok=True)

# Criando o arquivo __init__.py para transformar 'app' em um pacote Python
with open('app/__init__.py', 'w') as f:
    pass

### 3.2 a 3.4 - Construindo `app/main.py`

Vamos escrever o código completo da nossa API no arquivo `app/main.py`. Este arquivo irá:
1. Importar as bibliotecas e inicializar o FastAPI.
2. Carregar o modelo na inicialização.
3. Definir os modelos de dados para requisição (`ReviewRequest`) e resposta (`PredictionResponse`) com Pydantic.
4. Criar o endpoint `POST /predict_sentiment`.

In [22]:
%%writefile app/main.py
# Conteúdo de app/main.py

import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os

# Inicializa a aplicação FastAPI
app = FastAPI(
    title="Sentiment Analysis API",
    description="A simple API to predict sentiment from text.",
    version="0.1.0"
)

# Carrega o pipeline de ML na inicialização
pipeline = None
model_path = 'models/sentiment_pipeline.joblib'

@app.on_event("startup")
def load_model():
    global pipeline
    if os.path.exists(model_path):
        pipeline = joblib.load(model_path)
        print(f"Modelo carregado de {model_path}")
    else:
        print(f"Arquivo do modelo não encontrado em {model_path}")

# Define o modelo de dados para o corpo da requisição usando Pydantic
class ReviewRequest(BaseModel):
    text: str

# Define o modelo de dados para a resposta
class PredictionResponse(BaseModel):
    sentiment: str
    confidence: float

@app.post("/predict_sentiment", response_model=PredictionResponse)
def predict_sentiment(request: ReviewRequest):
    """
    Recebe um texto de review e retorna a predição de sentimento (Positivo/Negativo)
    e a confiança da predição.
    """
    if pipeline is None:
        raise HTTPException(status_code=503, detail="Modelo não está disponível")

    # 1. Extrai o texto da requisição
    review_text = request.text

    try:
        # 2. Passa o texto para o pipeline
        prediction_code = pipeline.predict([review_text])[0]
        prediction_proba = pipeline.predict_proba([review_text])[0]

        # 3. Mapeia o código da predição para um rótulo legível
        sentiment_map = {0: 'Negative', 1: 'Positive'}
        prediction_label = sentiment_map.get(int(prediction_code), 'Unknown')

        # 4. Obtém a confiança da predição (a maior probabilidade)
        confidence = float(max(prediction_proba))

        # 5. Retorna uma resposta JSON estruturada
        return PredictionResponse(sentiment=prediction_label, confidence=confidence)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Overwriting app/main.py


### 3.5 Exercício 2: Adicionar um Endpoint de Health Check (20 min)

**Tarefa:** É uma prática padrão em microsserviços ter um endpoint de "health check" para que sistemas de monitoramento (como Kubernetes ou um load balancer) possam verificar se a aplicação está funcionando corretamente.

1.  Adicione um novo endpoint à sua aplicação `app/main.py`.
2.  Ele deve responder a requisições `GET` no caminho (path) `/health`.
3.  A função do endpoint não deve receber nenhum argumento.
4.  Ela deve retornar um dicionário JSON simples, como `{"status": "ok"}`.

In [23]:
# Usamos o modo 'append' (-a) do %%writefile para adicionar o novo endpoint ao final do arquivo
%%writefile -a app/main.py

@app.get("/health")
def health_check():
    """Endpoint de Health Check para verificar se a API está online."""
    return {"status": "ok"}

Appending to app/main.py


---

## Parte 4: Passo a Passo da Prática: Executando e Testando a API

### 4.1 Instalar as dependências

Primeiro, criamos o arquivo `requirements.txt`.

In [24]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
scikit-learn
joblib
python-multipart
requests

Overwriting requirements.txt


Agora, instalamos as dependências usando `pip`.

In [25]:
!pip install -r requirements.txt

### 4.2 Iniciar o servidor web com `uvicorn`

**IMPORTANTE:** O comando abaixo irá iniciar um servidor web que **bloqueará a execução deste notebook**. Você deve executá-lo em um **terminal separado**, no mesmo diretório onde este notebook está salvo.

```bash
uvicorn app.main:app --reload
```

Após executar o comando no terminal, você verá uma saída indicando que o servidor está rodando em `http://127.0.0.1:8000`.

### 4.3 Usar a interface interativa do Swagger UI

Com o servidor rodando (a partir do seu terminal), abra seu navegador e acesse a seguinte URL:

**[http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)**

Siga os passos descritos no material da aula para testar os endpoints `/health` e `/predict_sentiment` interativamente.

### 4.4 (Bônus) Escrever um pequeno script cliente com `requests`

Vamos criar um script `client.py` para simular outro serviço consumindo nossa API. Podemos executá-lo diretamente do notebook.

In [26]:
%%writefile client.py
import requests
import json

api_url = "http://127.0.0.1:8000/predict_sentiment"

reviews = [
    "This movie was absolutely fantastic!",
    "The plot was predictable and the acting was subpar.",
    "I can't wait to see it again!",
    "A complete waste of my time and money."
]

for review_text in reviews:
    # Prepara os dados no formato JSON esperado pela API
    data = {"text": review_text}

    print(f"Enviando review: '{review_text}'")

    try:
        # Envia a requisição POST
        response = requests.post(api_url, json=data)

        # Verifica o resultado
        if response.status_code == 200:
            prediction = response.json()
            print(f"  -> Resposta da API: Sentimento = {prediction['sentiment']}, Confiança = {prediction['confidence']:.2f}")
        else:
            print(f"  -> Erro! Código de Status: {response.status_code}")
            print(f"     Resposta: {response.text}")
    except requests.exceptions.ConnectionError as e:
        print(f"  -> Erro de Conexão: Não foi possível conectar à API em {api_url}")
        print("     Certifique-se de que o servidor 'uvicorn' está em execução em um terminal separado.")
        break # Interrompe o loop se não conseguir conectar

    print("-" * 30)

Overwriting client.py


Agora, execute o cliente. **Lembre-se que o servidor `uvicorn` deve estar rodando em outro terminal.**

In [27]:
!python client.py

Enviando review: 'This movie was absolutely fantastic!'
  -> Erro de Conexão: Não foi possível conectar à API em http://127.0.0.1:8000/predict_sentiment
     Certifique-se de que o servidor 'uvicorn' está em execução em um terminal separado.


### 4.5 Exercício 3: Escrever um Cliente de Predição em Lote (20 min)

**Tarefa:** A API atual aceita uma review de cada vez. Em muitos cenários, seria mais eficiente enviar um lote de reviews e receber um lote de predições.

1.  **Modifique a API (`app/main.py`):**
    * Crie um novo modelo Pydantic `BatchReviewRequest` que espere uma lista de strings: `texts: list[str]`.
    * Crie um novo endpoint `POST /predict_sentiment_batch`.
    * Esta função receberá um `BatchReviewRequest`.
    * Chame `pipeline.predict()` e `pipeline.predict_proba()` na lista de textos inteira de uma vez.
    * A função deve retornar uma lista de `PredictionResponse`.
2.  **Modifique o Cliente (`client.py`):**
    * Altere o script para chamar o novo endpoint `/predict_sentiment_batch`.
    * Envie a lista inteira de reviews em uma única requisição POST.
    * Processe e imprima a lista de predições recebida.

In [28]:
# Solução Exercício 3 - Parte 1: Modificando a API
%%writefile -a app/main.py
from typing import List

class BatchReviewRequest(BaseModel):
    texts: List[str]

@app.post("/predict_sentiment_batch", response_model=List[PredictionResponse])
def predict_sentiment_batch(request: BatchReviewRequest):
    if pipeline is None:
        raise HTTPException(status_code=503, detail="Modelo não está disponível")

    try:
        predictions = pipeline.predict(request.texts)
        probas = pipeline.predict_proba(request.texts)

        results = []
        sentiment_map = {0: 'Negative', 1: 'Positive'}

        for pred, proba in zip(predictions, probas):
            sentiment = sentiment_map.get(int(pred), 'Unknown')
            confidence = float(max(proba))
            results.append(PredictionResponse(sentiment=sentiment, confidence=confidence))

        return results
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Appending to app/main.py


In [29]:
# Solução Exercício 3 - Parte 2: Modificando o Cliente
%%writefile client_batch.py
import requests

api_url = "http://127.0.0.1:8000/predict_sentiment_batch"

reviews = [
    "This movie was absolutely fantastic!",
    "The plot was predictable and the acting was subpar.",
    "I can't wait to see it again!",
    "A complete waste of my time and money."
]

data = {"texts": reviews}

print("Enviando lote de reviews para a API...")

try:
    response = requests.post(api_url, json=data)
    if response.status_code == 200:
        predictions = response.json()
        for original_review, prediction in zip(reviews, predictions):
            print(f"\nReview: '{original_review}'")
            print(f"  -> Predição: Sentimento={prediction['sentiment']}, Confiança={prediction['confidence']:.2f}")
    else:
        print(f"Erro! Código de Status: {response.status_code}")
        print(f"Resposta: {response.text}")
except requests.exceptions.ConnectionError:
    print(f"Erro de Conexão: Não foi possível conectar à API em {api_url}")

Overwriting client_batch.py


Execute o novo cliente em lote. **Lembre-se de reiniciar o servidor `uvicorn` no seu terminal** para que ele carregue as alterações que fizemos no arquivo `app/main.py`.

In [30]:
!python client_batch.py

Enviando lote de reviews para a API...
Erro de Conexão: Não foi possível conectar à API em http://127.0.0.1:8000/predict_sentiment_batch


---

## Parte 5: Projeto Final e Próximos Passos

### 5.1 Exercício Final: A API de Detecção de Spam

**Contexto:** Sua empresa precisa de um microsserviço que possa determinar se uma mensagem de texto enviada por um usuário é spam ou não ("ham").

**Dataset:** [Spam Text Message Classification](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

**Tarefa:** Utilizando as habilidades aprendidas neste workshop, você deve:

1.  **Exploração (Notebook):** Criar um Jupyter Notebook para carregar e explorar o dataset. Faça uma limpeza básica se necessário e confirme que um pipeline `TfidfVectorizer` + `LogisticRegression` é uma abordagem viável.
2.  **Treinamento e Serialização (Script Python):** Criar um script Python (`train_spam_detector.py`) que:
    * Carrega todo o dataset de spam.
    * Converte os rótulos "ham" e "spam" para `0` e `1`.
    * Define um pipeline com `TfidfVectorizer` e `LogisticRegression`.
    * Treina o pipeline com todos os dados.
    * Serializa o pipeline treinado para um arquivo `spam_pipeline.joblib`.
3.  **Construção da API (FastAPI):**
    * Criar uma nova aplicação FastAPI.
    * A aplicação deve carregar o `spam_pipeline.joblib` na inicialização.
    * Definir um endpoint `POST /predict_spam`.
    * Este endpoint deve aceitar um JSON com o texto da mensagem e retornar se é "spam" ou "ham", juntamente com a confiança da predição.
4.  **Teste (Swagger UI):**
    * Executar sua API com `uvicorn`.
    * Usar a documentação interativa da Swagger UI em `/docs` para testar seu endpoint.

### 5.2 O Caminho à Frente: Introdução a MLOps Avançado

O que foi construído hoje é a base de um sistema de ML em produção. A jornada para um sistema MLOps (Machine Learning Operations) completo envolve mais alguns passos importantes:

* **Containerização com Docker:** Empacotar nossa aplicação FastAPI, o modelo e suas dependências em uma imagem Docker. Isso cria um artefato portátil e consistente que pode ser executado em qualquer lugar.
* **CI/CD para ML (Integração e Implantação Contínuas):** Configurar pipelines automatizados que testam, constroem e implantam novas versões do modelo e da API após um `git push`.
* **Monitoramento de Modelos:** A performance de um modelo pode se degradar ao longo do tempo, um fenômeno conhecido como "model drift". Sistemas de MLOps avançados incluem monitoramento contínuo da acurácia do modelo e das distribuições dos dados de entrada para detectar o drift e acionar alertas para retreinamento.